In [6]:
import os
import tempfile
import shutil
from typing import List, Optional
from pathlib import Path
from settings import settings


from fastapi import FastAPI, UploadFile, File, HTTPException, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic_settings import BaseSettings
import json

# LlamaIndex Imports
from llama_index.core import Settings, StorageContext, VectorStoreIndex, SimpleDirectoryReader
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core.memory import ChatMemoryBuffer
import chromadb


In [ ]:
# Initialize FastAPI
app = FastAPI(title=settings.app_config.app_name)

# Enable CORS (Allows React frontend on localhost:3000 to talk to this backend)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # In production, replace "*" with specific domain
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

In [13]:
# Step 1: Import UUID
import uuid

# Step 2: Initialize the Session Store
# In-memory store for active user sessions
# Structure: { "session_id": { "chat_engine": engine_object, "filename": "doc.pdf" } }
active_sessions = {}

In [14]:
def initialize_engine(api_key: str, pdf_path: str):
    """
    Creates a NEW chat engine for a specific user.
    Returns the engine object so we can store it in the session dictionary.
    """
    # 1. Setup Models (Specific to this request)
    # We use the settings you imported earlier
    llm = GoogleGenAI(model="models/gemini-2.5-flash", api_key=api_key)
    embed_model = GoogleGenAIEmbedding(model="models/gemini-embedding-001", api_key=api_key)
    
    # Apply settings locally for this index creation
    from llama_index.core import Settings as LlamaSettings
    LlamaSettings.llm = llm
    LlamaSettings.embed_model = embed_model

    # 2. Setup Persistent ChromaDB
    # We use the path from your settings.py
    persistent_client = chromadb.PersistentClient(path=settings.app_config.db_path)
    
    # CRITICAL CHANGE: Create a unique collection name per session? 
    # For now, we will use one shared collection but isolate via logic, 
    # OR better: Use a unique collection name per session to prevent data mixing.
    # Let's generate a unique collection name based on a timestamp or random ID to be safe.
    import time
    unique_collection_name = f"collection_{int(time.time())}" 
    
    chroma_collection = persistent_client.get_or_create_collection(unique_collection_name)
    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)

    # 3. Load Documents
    documents = SimpleDirectoryReader(input_files=[pdf_path]).load_data()
    
    if not documents:
        raise ValueError("No text could be extracted from the PDF.")

    # 4. Create Index
    index = VectorStoreIndex.from_documents(
        documents, 
        storage_context=storage_context,
        show_progress=True
    )

    # 5. Create Chat Engine with Memory
    memory = ChatMemoryBuffer.from_defaults(token_limit=3900)
    
    chat_engine = index.as_chat_engine(
        chat_mode="context",
        memory=memory,
        system_prompt=(
            "You are a helpful assistant answering questions based strictly on the provided PDF context. "
            "If the answer is not in the context, say 'I cannot find this information in the document.' "
            "Always cite the page number if available."
        ),
    )
    
    # RETURN the engine instead of saving to a global variable
    return chat_engine, unique_collection_name

In [ ]:
@app.post("/upload")
async def upload_pdf(file: UploadFile = File(...), api_key: str = Form(...)):
    if not file.filename.endswith(".pdf"):
        raise HTTPException(status_code=400, detail="Only PDF files are allowed.")

    # 1. Generate a Unique Session ID for this user
    session_id = str(uuid.uuid4())
    
    # 2. Save uploaded file temporarily
    temp_dir = tempfile.mkdtemp()
    temp_file_path = os.path.join(temp_dir, file.filename)
    
    try:
        with open(temp_file_path, "wb") as buffer:
            shutil.copyfileobj(file.file, buffer)
        
        # 3. Initialize the engine (Get engine + collection name)
        chat_engine, collection_name = initialize_engine(api_key, temp_file_path)
        
        # 4. STORE in our dictionary
        active_sessions[session_id] = {
            "chat_engine": chat_engine,
            "collection_name": collection_name,
            "filename": file.filename
        }
        
        # 5. Return the Session ID to the user
        # The frontend MUST save this ID and send it with every chat request
        return {
            "message": "PDF processed successfully!", 
            "session_id": session_id,
            "filename": file.filename
        }
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Processing failed: {str(e)}")
    
    finally:
        # Cleanup temp file
        if os.path.exists(temp_file_path):
            os.remove(temp_file_path)
        if os.path.exists(temp_dir):
            os.rmdir(temp_dir)